#### Hybrid Search Langchain

In [1]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [2]:
import os
from pinecone import Pinecone, ServerlessSpec
index_name = "hybrid-search-langchain-pinecone"
pinecone_api_key = os.getenv("PINECONE_API_KEY")

## Initialize the pinecone client

pc = Pinecone()

## Create the index

if index_name not in pc.list_indexes().names():
  pc.create_index(
    name = index_name,
    dimension = 384, # Dimension of dense vector
    metric = 'dotproduct', # sparse values supported only for dotproduct
    spec = ServerlessSpec(cloud = "aws", region = "us-east-1"),
  )

In [3]:
index = pc.Index(index_name)
index

c:\Users\gargs\Downloads\LangchainProjects\5-Hybrid Search RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
## vector embedding and sparse matrix

import os 
from dotenv import load_dotenv
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [5]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder().default()
bm25_encoder

In [ ]:
sentences = [
  "In 2025, I visited New Zealand",
  "In 2024, I visited in Finland",
  "In 2023, I visited Norway",
]

## tfidf values on these sentences

bm25_encoder.fit(sentences)

## store the values to a json file

bm25_encoder.dump("bm25_values.json")

## Load to our BM25Encoder onject

bm25_encoder = BM25Encoder().load("bm25_values.json") 

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 31.54it/s]


In [7]:
retriever = PineconeHybridSearchRetriever(embeddings = embeddings, sparse_encoder = bm25_encoder, index = index)

In [8]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000001F175C7FB90>, index=<pinecone.db_data.index.Index object at 0x000001F12AE7BD90>)

In [9]:
retriever.add_texts(
  [
    "In 2025, I visited New Zealand",
    "In 2024, I visited Finland",
    "In 2023, I visited Norway",
  ]
)

100%|██████████| 1/1 [00:06<00:00,  6.10s/it]


In [10]:
retriever.invoke("Which city I visit first")

[Document(metadata={'score': 0.219899178}, page_content='In 2024, I visited Finland'),
 Document(metadata={'score': 0.20382452}, page_content='In 2023, I visited Norway'),
 Document(metadata={'score': 0.201077521}, page_content='In 2025, I visited New Zealand')]